In [2]:
%pip install --only-binary=:all: psycopg2-binary



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# %pip install boto3
# %pip install logger

### importing libraries

In [4]:
import boto3
import pandas as pd
import json
import psycopg2
import logger


### Reading configuration file

In [5]:
import configparser
config = configparser.ConfigParser()
config_path = '/Users/kamlesh/Desktop/Projects/S3-Redshift DE project/Configs/confirguration.config'

config.read(config_path)

print(config.sections())  # Check whether AWS is detected


['AWS', 'DWH']


### Access and Secret key and other configs

In [6]:
# fetch AWS Access and Secret key for aws-user account
KEY = config['AWS']['KEY']
SECRET = config.get('AWS','SECRET')



# Cluster/ DWH Configurations
DWH_CLUSTER_TYPE = config['DWH']['DWH_CLUSTER_TYPE']
DWH_NUM_NODES = config['DWH']['DWH_NUM_NODES']
DWH_NODE_TYPE = config['DWH']['DWH_NODE_TYPE']
DWH_CLUSTER_IDENTIFIER = config['DWH']['DWH_CLUSTER_IDENTIFIER']
DWH_DB = config['DWH']['DWH_DB']
DWH_DB_USER = config['DWH']['DWH_DB_USER']
DWH_DB_PASSWORD = config['DWH']['DWH_DB_PASSWORD']
DWH_PORT = config['DWH']['DWH_PORT']

# IAM Role
DWH_IAM_ROLE_NAME = config['DWH']['DWH_IAM_ROLE_NAME']

In [7]:
ec2 = boto3.resource('ec2',
                     region_name = 'ap-south-1',
                     aws_access_key_id = KEY,
                     aws_secret_access_key= SECRET
                     )

In [8]:
s3 = boto3.resource('s3',
                     region_name = 'ap-south-1',
                     aws_access_key_id = KEY,
                     aws_secret_access_key= SECRET
                     )

In [9]:
iam = boto3.client('iam',
                     region_name = 'ap-south-1',
                     aws_access_key_id = KEY,
                     aws_secret_access_key= SECRET
                     )

In [10]:
redshift = boto3.client('redshift',
                     region_name = 'ap-south-1',
                     aws_access_key_id = KEY,
                     aws_secret_access_key= SECRET
                     )

In [11]:
# Role ARN to fetch details
iamRoleArn = iam.get_role(RoleName = DWH_IAM_ROLE_NAME)['Role']['Arn']

In [12]:
# List Bucket
bucket  = s3.Bucket('amzn-s3-redshift-etl-pipeline')

# List all the files present in the S3 bucket
log_data_files  = [filename.key for filename in bucket.objects.filter(Prefix="")]
log_data_files

['allevents_pipe.txt',
 'allusers_pipe.txt',
 'category_pipe.txt',
 'date2008_pipe.txt',
 'listings_pipe.txt',
 'sales_tab.txt',
 'venue_pipe.txt']

# Creating Redshift Cluster

In [ ]:
try:
    response = redshift.create_cluster(
    ClusterType=DWH_CLUSTER_TYPE,
    NodeType=DWH_NODE_TYPE,

    # Identifiers and credentials
    DBName=DWH_DB,
    ClusterIdentifier=DWH_CLUSTER_IDENTIFIER,
    MasterUsername=DWH_DB_USER,
    MasterUserPassword=DWH_DB_PASSWORD,
    
    #Roles for S3 access
    IamRoles=
    [iamRoleArn]
    )

    print(f"Cluster is created {DWH_CLUSTER_IDENTIFIER}")
except Exception as e:
    print(e)
    raise e


### Fetching cluster details

In [14]:
def clusterProps(props):
    pd.set_option('display.max_colwidth',None)
    keysToShow = ['ClusterIdentifier','NodeType','ClusterStatus','MasterUsername','DBName','Endpoint', 'VpcId']

    x = [ (k,v) for k,v in props.items() if k in keysToShow]

    return pd.DataFrame(x,columns=['Key','Value'])


cluster_properties = redshift.describe_clusters(ClusterIdentifier= DWH_CLUSTER_IDENTIFIER)['Clusters'][0]

clusterProps(cluster_properties)

,Key,Value
0,ClusterIdentifier,s3-redshift-project
1,NodeType,ra3.large
2,ClusterStatus,creating
3,MasterUsername,kamleshaws
4,DBName,s3_redshift_project
5,VpcId,vpc-008712917656c3537


In [15]:
DWH_ENDPOINT = cluster_properties['Endpoint']['Address']
DWH_ROLE_ARN = cluster_properties['IamRoles'][0]['IamRoleArn']
DB_NAME = cluster_properties['DBName']
DB_USER = cluster_properties['MasterUsername']

KeyError: 'Endpoint'

### Allowing all traffic on to the VPC

In [ ]:
try:
    vpc = ec2.Vpc(id=cluster_properties['VpcId'])
    defaultSg = list(vpc.security_groups.filter(Filters=[{'Name': 'group-name', 'Values': ['default']}]))[0]
    print(defaultSg)

    defaultSg.authorize_ingress(
        GroupName=defaultSg.group_name,
        CidrIp='0.0.0.0/0',
        IpProtocol='tcp',
        FromPort=int(DWH_PORT),
        ToPort=int(DWH_PORT)
    )

except Exception as e:
    print(e)
    # raise e


ec2.SecurityGroup(id='sg-0d3417d0c2b7bda08')
An error occurred (InvalidPermission.Duplicate) when calling the AuthorizeSecurityGroupIngress operation: the specified rule "peer: 0.0.0.0/0, TCP, from port: 5439, to port: 5439, ALLOW" already exists


### Connecting to the Redshift datawarehouse

In [ ]:
try:
    conn = psycopg2.connect(host= DWH_ENDPOINT,dbname = DB_NAME,user = DB_USER,password = DWH_DB_PASSWORD,port= int(DWH_PORT))
except Exception as e:
    print("Error: Could not make connection to the postgres database")
    print(e)
    # raise e

Error: Could not make connection to the postgres database
connection to server at "s3-redshift-project.cyyqauj3idkn.ap-south-1.redshift.amazonaws.com" (172.31.1.54), port 5439 failed: Operation timed out
	Is the server running on that host and accepting TCP/IP connections?



### Accessing dwh using cursor

In [ ]:
try:
    cur = conn.cursor()
except psycopg2.Error as e:
    print("Error: Could not get curser to the database")
    print(e)

AttributeError: module 'psycopg_binary' has no attribute 'Error'

In [ ]:
# try:
#     cur.execute(
#         """
#         CREATE TABLE Users(
#         userid integer not null distkey sortkey,
#         username char(8),
#         firstname varchar(30),
#         lastname varchar(30),
#         city varchar(30),
#         state char(2),
#         email varchar(100),
#         phone char(14),
#         likesports boolean,
#         liketheatre boolean,
#         like
#         )
#         """
#     )